#Dézippe et charge les fichiers dans Colab

In [ ]:
import zipfile
import os

# Dézipper le fichier
with zipfile.ZipFile("ml-latest-small.zip", "r") as zip_ref:
    zip_ref.extractall("ml-latest-small")

# Vérifier les fichiers extraits
os.listdir("ml-latest-small")

['tags.csv', 'links.csv', 'README.md', 'movies.csv', 'ratings.csv']

# Création de la matrice


In [ ]:
import pandas as pd
import numpy as np

# Charger le fichier des notes
ratings = pd.read_csv("ml-latest-small/ratings.csv")

# Création de la matrice d'utilité
utility_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')
utility_matrix.fillna(0, inplace=True)  # Remplir les valeurs NaN avec 0

#Calcul de la similarité

Similitude Cosinus et Corrélation de Pearson

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Similarité entre deux utilisateurs (ex : user 1 et user 2)
cosine_sim_users = cosine_similarity(utility_matrix)
print("Similarité Cosinus entre user 1 et user 2 :", cosine_sim_users[0, 1])

# Similarité entre deux films (ex : movie 1 et movie 2)
cosine_sim_movies = cosine_similarity(utility_matrix.T)
print("Similarité Cosinus entre movie 1 et movie 2 :", cosine_sim_movies[0, 1])

# Corrélation entre deux utilisateurs
pearson_sim_users = utility_matrix.T.corr(method='pearson')
print("Corrélation Pearson entre user 1 et user 2 :", pearson_sim_users.iloc[0, 1])

# Corrélation de Pearson entre deux films spécifiques (par exemple, movie 1 et movie 2)
movie_1_id = 1  # ID du premier film
movie_2_id = 2  # ID du second film

# Extraire les évaluations des films
movie_1_ratings = utility_matrix[movie_1_id]
movie_2_ratings = utility_matrix[movie_2_id]

# Calcul de la corrélation de Pearson
pearson_corr = movie_1_ratings.corr(movie_2_ratings, method='pearson')

print("Corrélation Pearson entre movie 1 et movie 2 :", pearson_corr)

Similarité Cosinus entre user 1 et user 2 : 0.027282865283323236
Similarité Cosinus entre movie 1 et movie 2 : 0.41056206350173163
Corrélation Pearson entre user 1 et user 2 : 0.019399824956085835
Corrélation Pearson entre movie 1 et movie 2 : 0.23132650440398173


Interprétation des résultat obtenus :

User 1 et User 2 ont une très faible similarité Cosinus (0.027) et une corrélation Pearson presque nulle (0.019). Cela signifie qu'ils ont des préférences très différentes et qu’il n’y a quasiment aucune relation entre leurs évaluations.

Movie 1 et Movie 2 ont une similarité Cosinus modérée (0.411) et une corrélation Pearson plus faible (0.231). Cela indique qu’ils partagent certaines similitudes dans les évaluations des utilisateurs, mais la relation reste relativement faible.


Conclusion :
Les utilisateurs n'ont presque rien en commun dans leurs goûts, tandis que les films ont un certain degré de ressemblance dans les avis reçus, mais sans être fortement corrélés (sans être considérés comme très proches non plus).

# Représentation TF-IDF des descriptions

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Charger les films
movies = pd.read_csv("ml-latest-small/movies.csv")

# Ajouter une colonne fictive de descriptions (si nécessaire)
movies["description"] = "This is a placeholder description for movie " + movies["title"]

# Appliquer TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["description"])


# Recommandation basé sur le contenu

In [ ]:
from sklearn.metrics.pairwise import linear_kernel

# Calculer la similarité des films
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Fonction de recommandation
def recommend_movies(movie_title, movies, cosine_sim):
    idx = movies[movies["title"] == movie_title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]  # Top 5
    movie_indices = [i[0] for i in sim_scores]
    return movies.iloc[movie_indices]["title"]

# Tester la recommandation
recommend_movies("Jumanji (1995)", movies, cosine_sim)

,title
9636,Jumanji: Welcome to the Jungle (2017)
26,Now and Then (1995)
529,Two Much (1995)
178,Wild Bill (1995)
41,To Die For (1995)


# Avantages et limites

Avantages :

Ne nécessite pas les notes des utilisateurs, donc fonctionne bien pour les nouveaux utilisateurs.

Permet de recommander des films même s’ils n’ont pas été notés.



Limitations :

Ne prend pas en compte les préférences personnelles.

Limité par la qualité des descriptions.

Difficulté à recommander des films très différents mais appréciés par les mêmes utilisateurs.